# 实验：GPT-2 124M × FineWeb-Edu (sample-10BT)

本实验使用 **FineWeb-Edu `sample-10BT`** 子集对 **GPT-2 124M** 进行预训练。  
FineWeb-Edu 是从 CommonCrawl 过滤出的高质量英文教育文本，`sample-10BT` 含约 100 亿 tokens，
是验证 124M 量级模型预训练效果的常用基准数据集。

## 你将依次完成的 6 步

1. **环境准备**：确认工作目录与依赖路径。
2. **实验配置**：设置数据规模、训练步数等核心参数。
3. **下载语料**：从 HF 拉取 FineWeb-Edu sample-10BT（按 shard 分批下载）。
4. **自训练分词器**：在语料子集上训练字节级 BPE 分词器，保存为 `tokenizer.json`。
5. **启动训练**：配置适合 124M 模型的超参数并运行训练循环。
6. **推理验证**：加载 checkpoint，生成文本检验效果。

---

> **GPU 显存参考**  
> - `batch_size=4, block_size=1024, grad_accum_steps=32` → 约 6–8 GB  
> - `batch_size=8, block_size=1024, grad_accum_steps=16` → 约 12–16 GB  
> - 若在 CPU / MPS 上运行，建议将 `block_size` 降至 256，`batch_size` 降至 2。

## (可选) HF Token 配置

设置 `HF_TOKEN` 可以提升下载速率限制，避免 rate-limit 报错（特别是在国内镜像节点访问频繁时）。  
Token 在 [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 生成（Read 权限即可）。  
若不需要也可跳过本 cell。

In [1]:
import os

# 在此填入你的 HF Token（留空则不设置）
HF_TOKEN = ""   # 示例: "hf_xxxxxxxxxxxxxxxx"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    # 同时让 huggingface_hub 全局使用该 token
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ HF Token 已设置")
else:
    print("ℹ 未设置 HF Token，使用匿名访问（可能受速率限制）")

ℹ 未设置 HF Token，使用匿名访问（可能受速率限制）


## Step 1. 环境准备

In [2]:
import os
import sys
from pathlib import Path

# 自动向上查找项目根目录（以 pyproject.toml 作为标志文件）
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), None)

if repo_root is None:
    print("⚠️ 未自动定位到项目根目录。")
    print("请手动执行: %cd d:/codingProgram/LLM-Walk-Through")
    raise RuntimeError("请先切换到仓库根目录后再继续。")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("当前工作目录:", Path.cwd())
print("根目录检测:", (Path("pyproject.toml").exists(), Path("core").exists()))

当前工作目录: D:\codingProgram\LLM-Walk-Through
根目录检测: (True, True)


## Step 2. 实验配置

根据你的 GPU 显存和可接受的训练时长调整以下参数：

| 参数 | 说明 |
|------|------|
| `NUM_SHARDS` | 下载 FineWeb-Edu sample-10BT 的 shard 数（该子集共 **14** 个 shard） |
| `MAX_SAMPLES` | 额外限制最多读取样本数（`None` 表示不限） |
| `MAX_STEPS` | 训练步数；参考 Chinchilla 规律，124M 模型最优约 **20_000** 步 |
| `BATCH_SIZE` | 单卡 batch size，与 `GRAD_ACCUM` 组合控制有效 batch |
| `GRAD_ACCUM` | 梯度累积步数；有效 batch tokens = `BATCH_SIZE × GRAD_ACCUM × BLOCK_SIZE` |

In [3]:
# ── 数据配置 ─────────────────────────────────────────────────────────────────
REPO_ID      = "HuggingFaceFW/fineweb-edu"
SUBSET_NAME  = "sample-10BT"           # FineWeb-Edu 子集名称
SPLIT        = "train"
TEXT_FIELD   = "text"                  # FineWeb-Edu 文本字段
NUM_SHARDS   = 5                        # 下载前 N 个 shard（共 14 个）
                                        # 4~5 个 shard ≈ 3.5B tokens，已满足 124M 最优训练量
MAX_SAMPLES  = None                     # None = 不限（使用所有已下载样本）
MAX_CHARS    = None                     # None = 不限字符数
HF_ENDPOINT  = "https://hf-mirror.com"

CACHE_DIR    = Path("data/cache/fineweb_edu_10bt")

# ── 分词器 ────────────────────────────────────────────────────────────────────
TOKENIZER_KIND = "byte_bpe"           # 字节级 BPE，与 GPT-2 算法一致，在语料上自训练
VOCAB_SIZE     = 8192                 # 目标词表大小（可调整：4k~32k 均可）

# ── 训练超参数 ─────────────────────────────────────────────────────────────────
# 参考 Chinchilla 规律：124M 模型最优训练量 ≈ 20 × 参数量 = 2.5B tokens
# 有效 batch = 128，每步 131K tokens → 20_000 步 ≈ 2.6B tokens
OUT_DIR         = "runs/fineweb_edu_124m"
MAX_STEPS       = 20_000             # Chinchilla optimal；快速验证可设为 2_000
BATCH_SIZE      = 4                   # 单卡 batch size（8GB 显存建议保持 4）
GRAD_ACCUM      = 32                  # 有效 batch = 4×32 = 128，tokens/step ≈ 131K
BLOCK_SIZE      = 1024                # GPT-2 标准上下文窗口
LEARNING_RATE   = 6e-4                # GPT-2 124M 论文推荐峰值学习率
MIN_LR          = 6e-5                # cosine decay 终点（峰值的 10%）
WARMUP_STEPS    = 2_000               # 占 20K 总步数的 10%，比例合理
EVAL_INTERVAL   = 1_000
EVAL_ITERS      = 20
LOG_INTERVAL    = 50
WEIGHT_DECAY    = 0.1
BETA1, BETA2    = 0.9, 0.95
GRAD_CLIP       = 1.0
SEED            = 42

# ── 显示有效 batch tokens ─────────────────────────────────────────────────────
effective_tokens = BATCH_SIZE * GRAD_ACCUM * BLOCK_SIZE
print(f"有效 batch size: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} samples")
print(f"有效 tokens/step: {effective_tokens:,}")
print(f"总训练 tokens: {effective_tokens * MAX_STEPS:,}")
print(f"数据 shard 数: {NUM_SHARDS} / 14（建议 4~5 个即可满足 Chinchilla optimal）")

有效 batch size: 4 × 32 = 128 samples
有效 tokens/step: 131,072
总训练 tokens: 2,621,440,000
数据 shard 数: 5 / 14（建议 4~5 个即可满足 Chinchilla optimal）


## Step 3. 下载 FineWeb-Edu 语料

FineWeb-Edu `sample-10BT` 共有 **14 个 parquet shard**（`sample/10BT/000_00000.parquet` ~ `013_00000.parquet`），
通过 `num_shards` 参数精确控制只下载前 `NUM_SHARDS` 个 shard，避免拉取全量数据集。

> **关于数据量**：`sample-10BT` 子集约含 **100 亿 tokens**，远超过 124M 模型所需。
> 按 Chinchilla 规律，124M 模型最优训练量约 **25 亿 tokens**，即约 **4 个 shard** 即可满足。
> 已下载的 shard 若未用完，会在训练时通过随机采样复用（不会浪费显存）。

> 首次运行会连接 `hf-mirror.com` 下载数据，请确保网络畅通。  
> 若本地已有足够的 shard 则自动跳过。

In [ ]:
import os
from data.download import download

if HF_ENDPOINT:
    os.environ["HF_ENDPOINT"] = HF_ENDPOINT

# ── Step 3: 下载 shard ───────────────────────────────────────────────────────
# 使用 num_shards 参数自动列出并下载前 N 个 shard（已存在的 shard 不会重复下载）
snapshot_dir = download(
    repo_id=REPO_ID,
    local_dir=CACHE_DIR,
    subset_name=SUBSET_NAME,
    num_shards=NUM_SHARDS,
    hf_endpoint=HF_ENDPOINT,
)

total_size_mb = sum(p.stat().st_size for p in snapshot_dir.rglob("*.parquet")) / 1024 / 1024
parquet_count = sum(1 for _ in snapshot_dir.rglob("*.parquet"))
print(f"\n数据目录: {snapshot_dir}")
print(f"已缓存 parquet shard 数: {parquet_count}")
print(f"总大小: {total_size_mb:.1f} MB")

## Step 4. 自训练字节级 BPE 分词器

FineWeb-Edu 是英文语料，使用项目自实现的 **ByteBPETokenizer**（字节级 BPE），
算法与 GPT-2 完全一致（先 utf-8 字节化，再做 BPE 合并），但**词表在本语料上自训练**。  
训练时会自动从 parquet 中采样最多 5000 万字符，训练完成后保存到 `tokenizer.json`，
后续运行可直接复用，无需重新训练。

In [ ]:
import tqdm
from core.tokenizer import ByteBPETokenizer, load_tokenizer
from data.encode import iter_texts, encode_corpus

tokenizer_path = CACHE_DIR / "tokenizer.json"

if tokenizer_path.exists():
    tok = load_tokenizer(tokenizer_path)
    print(f"复用已有分词器 kind={tok.KIND!r}: {tokenizer_path}")
else:
    print("采样语料用于分词器训练...")
    sample_texts = []
    sample_chars = 0
    max_train_chars = 50_000_000
    for text in tqdm.tqdm(iter_texts(CACHE_DIR, max_chars=max_train_chars), desc="Sampling", unit="sample"):
        sample_texts.append(text)
        sample_chars += len(text)
        if sample_chars >= max_train_chars:
            break
    print(f"已采样 {len(sample_texts)} 条样本，共 {sample_chars:,} 字符")

    tok = ByteBPETokenizer.train("\n\n".join(sample_texts), vocab_size=VOCAB_SIZE, verbose=True)
    tok.save(tokenizer_path)
    print(f"分词器训练完成，词表大小: {tok.vocab_size}")

result = encode_corpus(
    cache_dir=CACHE_DIR,
    tokenizer=tok,
    val_ratio=0.05,
)

print("分词 & 编码完成:")
for k, v in result.items():
    print(f"  {k}: {v}")

import numpy as np
train_tokens = np.memmap(result["train_bin"], dtype=result["dtype"], mode="r")
val_tokens   = np.memmap(result["val_bin"],   dtype=result["dtype"], mode="r")
print(f"训练集 tokens: {len(train_tokens):,}")
print(f"验证集 tokens: {len(val_tokens):,}")
print(f"词表大小: {result['vocab_size']}")

## Step 5. 配置与启动训练

### 超参数说明

GPT-2 124M 的训练配置参考 [Chinchilla](https://arxiv.org/abs/2203.15556) 和原始 GPT-2 论文：

| 参数 | 本实验值 | 说明 |
|------|----------|------|
| 学习率 | `6e-4` | GPT-2 124M 推荐峰值 |
| cosine decay 下界 | `6e-5` | 峰值的 10% |
| warmup steps | `2_000` | 占 20K 总步数的 10%，比例合理 |
| 有效 batch (tokens) | `128 × 1024 = 131K` | 单卡可承受范围 |
| weight decay | `0.1` | AdamW 正则化 |
| gradient clip | `1.0` | 稳定训练 |

### Chinchilla 训练量计算

124M 模型参数 × 20 = **2.5B tokens**（Chinchilla 推荐最优值）。

当前配置 `BATCH_SIZE=4, GRAD_ACCUM=32, BLOCK_SIZE=1024`：
- 每步 tokens = `4 × 32 × 1024 = 131,072`
- 达到 2.5B tokens 需要 `2.5B ÷ 131K ≈ 19,000` 步
- 本实验设 `MAX_STEPS=20_000`，刚好达到 Chinchilla optimal

### 数据量与 shard 数

`sample-10BT` 子集约 10B tokens，但 124M 模型只需 2.5B 即可收敛：
- **推荐**：下载 **4~5 个 shard**（约 3.5B tokens），配合 20K 步训练
- 14 个 shard 全部下载也可以，但训练时只会随机采样其中约 1/4 的数据
- 如果已经下载了全部 14 个 shard，保持 `NUM_SHARDS=14` 即可，不影响训练正确性

### 不同场景的快速配置

| 目标 | NUM_SHARDS | MAX_STEPS | 总 tokens | 预估时间（单卡 A100） |
|------|-----------|-----------|-----------|---------------------|
| 流程验证 | 2 | 1_000 | 131M | ~15 分钟 |
| 快速收敛 | 5 | 20_000 | 2.6B | ~4–6 小时 |
| 充分训练 | 14 | 40_000 | 5.2B | ~8–12 小时 |

> 若在 CPU / MPS 上运行，建议将 `block_size` 降至 256，`batch_size` 降至 2。

In [6]:
from omegaconf import OmegaConf
from core.utils.config import load_config

# 以 pretrain_tiny.yaml 作为基础结构，用 overrides 覆盖所有关键参数
_base_cfg = OmegaConf.load("configs/train/pretrain_tiny.yaml")

overrides = [
    # ── 模型 ──────────────────────────────────────
    "defaults.model=configs/model/gpt2_124m.yaml",

    # ── 数据 ──────────────────────────────────────
    f"data.cache_dir={CACHE_DIR}",
    f"data.hf.repo_id={REPO_ID}",
    f"data.hf.subset_name={SUBSET_NAME}",
    f"data.hf.split={SPLIT}",
    f"data.hf.text_field={TEXT_FIELD}",
    "data.hf.max_samples=null",
    "data.hf.max_chars=null",
    "data.hf.val_ratio=0.05",
    f"data.hf.num_shards={NUM_SHARDS}",
    f"data.tokenizer.kind={TOKENIZER_KIND}",
    f"data.tokenizer.vocab_size={VOCAB_SIZE}",

    # ── 训练 ──────────────────────────────────────
    f"train.out_dir={OUT_DIR}",
    f"train.batch_size={BATCH_SIZE}",
    f"train.block_size={BLOCK_SIZE}",
    f"train.grad_accum_steps={GRAD_ACCUM}",
    f"train.max_steps={MAX_STEPS}",
    f"train.eval_interval={EVAL_INTERVAL}",
    f"train.eval_iters={EVAL_ITERS}",
    f"train.log_interval={LOG_INTERVAL}",
    f"train.learning_rate={LEARNING_RATE}",
    f"train.min_lr={MIN_LR}",
    f"train.warmup_steps={WARMUP_STEPS}",
    f"train.weight_decay={WEIGHT_DECAY}",
    f"train.beta1={BETA1}",
    f"train.beta2={BETA2}",
    f"train.grad_clip={GRAD_CLIP}",
    f"train.seed={SEED}",
    "train.lr_schedule=cosine",
    "train.dtype=auto",
    "train.amp=true",
]

train_cfg = load_config("configs/train/pretrain_tiny.yaml", overrides=overrides)

print("训练配置摘要:")
print(f"  模型:          gpt2_124m  (12层, 12头, d_model=768, vocab={VOCAB_SIZE})")
print(f"  数据集:        {REPO_ID} / {SUBSET_NAME}")
print(f"  分词器:        {TOKENIZER_KIND}")
print(f"  batch_size:    {BATCH_SIZE}  grad_accum={GRAD_ACCUM}  → 有效 batch={BATCH_SIZE*GRAD_ACCUM}")
print(f"  block_size:    {BLOCK_SIZE}")
print(f"  max_steps:     {MAX_STEPS:,}")
print(f"  总训练 tokens: {BATCH_SIZE * GRAD_ACCUM * BLOCK_SIZE * MAX_STEPS:,}")
print(f"  学习率:        {LEARNING_RATE} → {MIN_LR} (cosine, warmup={WARMUP_STEPS})")
print(f"  输出目录:      {OUT_DIR}")

训练配置摘要:
  模型:          gpt2_124m  (12层, 12头, d_model=768, vocab=8192)
  数据集:        HuggingFaceFW/fineweb-edu / sample-10BT
  分词器:        byte_bpe
  batch_size:    4  grad_accum=32  → 有效 batch=128
  block_size:    1024
  max_steps:     20,000
  总训练 tokens: 2,621,440,000
  学习率:        0.0006 → 6e-05 (cosine, warmup=2000)
  输出目录:      runs/fineweb_edu_124m


In [ ]:
from train.pretrain import train as run_train

# 等价终端命令（可复制后在终端中断点续训）：
print("等价终端命令:")
print(f"  python -m train.pretrain --config configs/train/pretrain_tiny.yaml \\")
print(f"      data.cache_dir={CACHE_DIR} \\")
print(f"      data.tokenizer.kind={TOKENIZER_KIND} \\")
print(f"      train.out_dir={OUT_DIR} \\")
print(f"      train.max_steps={MAX_STEPS}")
print()
print("▶ 开始训练 ...")
run_train(train_cfg)

等价终端命令:
  python -m train.pretrain --config configs/train/pretrain_tiny.yaml \
      data.cache_dir=data\cache\fineweb_edu_10bt \
      data.tokenizer.kind=byte_bpe \
      train.out_dir=runs/fineweb_edu_124m \
      train.max_steps=20000

▶ 开始训练 ...
[setup] device=cuda dtype=torch.bfloat16 amp=True world_size=1
[data] 本地已有 13 个 parquet shard >= 目标 5，跳过远程探测与下载
[data] 完成：本地 13 个 parquet shard，元数据已写入 data\cache\fineweb_edu_10bt\data_meta.json
[data] 复用已有分词器 kind='byte_bpe': data\cache\fineweb_edu_10bt\tokenizer.json
[data] bin 文件已存在，跳过重新编码。
[setup] model params = 92.13M


训练:   0%|          | 0/20001 [00:00<?, ?step/s]

[eval] step=0 train=9.2058 val=9.2090
[eval] step=1000 train=4.2606 val=4.2771
[eval] step=2000 train=3.5004 val=3.5298
[eval] step=3000 train=3.3140 val=3.3452
[eval] step=4000 train=3.1556 val=3.1818
[eval] step=5000 train=3.1611 val=3.1228
[eval] step=6000 train=3.0359 val=3.0316
[eval] step=7000 train=3.0475 val=3.0639
[eval] step=8000 train=3.0096 val=2.9876
[eval] step=9000 train=2.9612 val=2.9817


## Step 6. 推理验证

训练完成后，使用保存的 checkpoint 生成文本。  
FineWeb-Edu 是英文教育内容，prompt 请使用英文。

In [ ]:
ckpt_path = f"{OUT_DIR}/ckpt.pt"
prompt = "The study of mathematics requires"

!python -m scripts.generate \
    --checkpoint {ckpt_path} \
    --prompt "{prompt}" \
    --max-new-tokens 200 \
    --temperature 0.8 \
    --top-k 50

---

## 常见问题 & 进阶提示

### 数据
- **如何增加训练数据**：增大 `NUM_SHARDS`（最多 14），或将 `MAX_SAMPLES` 设为 `None`。
  但注意 124M 模型 Chinchilla optimal 只需 2.5B tokens，约 4 个 shard 即足够。
- **下载很慢**：检查 `HF_ENDPOINT`，也可换成 `https://huggingface.co`（直连）。
- **重置缓存**：删除 `data/cache/fineweb_edu_10bt/` 整个目录后重新执行 Step 3。

### 训练
- **OOM / 显存不足**：减小 `BATCH_SIZE`，相应增大 `GRAD_ACCUM` 保持有效 batch 不变。  
  例如：`BATCH_SIZE=2, GRAD_ACCUM=64`。
- **加快训练速度**：确认 `dtype=auto` 会在 CUDA 上选择 `bfloat16`，PyTorch SDPA 会自动启用 Flash 内核。
- **多卡训练**：在终端使用 `torchrun --nproc_per_node=N -m train.pretrain --config ... distributed.backend=ddp`。
- **断点续训**：目前训练脚本每次从头开始；如需续训，可修改 `train/pretrain.py` 加载已有 checkpoint。

### 评估
- `eval loss` 收敛参考：2K 步后约 `3.5–4.0`；20K 步后约 `2.8–3.2`。
- 与 GPT-2 官方对比时注意：官方使用 WebText 训练，本实验用 FineWeb-Edu，分布不同。